# Super-Resolution with U-Net
---

## Motivation
Many numerical models provide predictions at a spatial resolution that is too coarse for some applications. Super-resolution aims to reconstruct a high-resolution field from a lower-resolution representation.

> **Goal** \
> Can a neural network learn to reconstruct a high-resolution volcanic ash concentration field from a low-resolution one?

* We will use an ensemble of FALL3D simulations as our dataset. Each ensemble member is a 2-D volcanic ash concentration field
* The original high-resolution field is our target, while the artificially degraded field is the input to the neural network

### The super-resolution task

For this example, we will artificially reduce the resolution by a factor of 4. 
The neural network will therefore learn the mapping:
$$ X_{LR} \to X_{HR} $$

## &#128221; Example 3.1: loading training data

We load the FALL3D ensemble containing 128 simulations of the volcanic ash field.

In [ ]:
### Get data
from modules.helper import download_file

fname_train = download_file("https://saco.csic.es/public.php/dav/files/YD59Ye7cBx79Prt")
fname_val   = download_file("https://saco.csic.es/public.php/dav/files/7K8GGqoM4zEAemj")

In [ ]:
import xarray as xr

ds = xr.open_dataset(fname_train)
da_train = ds["tephra_col_mass"]

ds = xr.open_dataset(fname_val)
da_val = ds["tephra_col_mass"]

In [ ]:
from modules.dataset import SuperResolutionDataset, LogTransform

ds_conf = dict(
    scale_factor = (4,8),
    transform    = LogTransform()
)

train_dataset = SuperResolutionDataset(da_train, **ds_conf)
val_dataset   = SuperResolutionDataset(da_val, **ds_conf)

In [ ]:
X_LR, X_HR = train_dataset[12]
X_LR.shape, X_HR.shape

We will use the ens dimension as independent training samples.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].pcolormesh(X_LR[0])
axes[0].set_title("Low resolution")

im = axes[1].pcolormesh(X_HR[0])
axes[1].set_title("High resolution")

fig.colorbar(im)

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=16)

## Define the model

In [ ]:
from modules.model import SuperResolutionUNet

model = SuperResolutionUNet(scale_factor=(4, 8))

In [ ]:
from torchsummary import summary

# Instantiate model
summary(model, input_size=(1, 32, 32))

## Training

In [ ]:
import torch
import torch.nn as nn
from modules.training import SupervisedTrainer

trainer = SupervisedTrainer(
    model=model,
    criterion=nn.MSELoss(),
    optimizer=torch.optim.Adam(model.parameters(), lr=1e-3),
)

In [ ]:
history = trainer.fit(train_loader, val_loader, epochs=100, log_every=10)

In [ ]:
history.plot(
    x="epoch",
    y=["train_loss", "val_loss"],
)

## Inference

In [ ]:
model.eval()

predictions = []
targets = []
inputs = []

with torch.no_grad():
    for X, y in val_loader:
        predictions.append(model(X))
        inputs.append(X)
        targets.append(y)

X_val = torch.cat(inputs)
y_pred = torch.cat(predictions)
y_val = torch.cat(targets)

# Back to physical units
if False:
    X_val = transform.invert(X_val.numpy())
    y_pred = transform.invert(y_pred.numpy())
    y_val = transform.invert(y_val.numpy())

In [ ]:
print(y_val.min(), y_val.max())
print(y_pred.min(), y_pred.max())

In [ ]:
import numpy as np
n = len(X_val)
nplots = 6

indexes = np.random.choice(n, size=nplots, replace=False)

fig, axes = plt.subplots(nplots, 3, figsize=(9, 3 * nplots))

conf_plt = {'vmin':0, 'vmax': 8}

for ip in range(nplots):
    i = indexes[ip]
    axes[ip, 0].pcolormesh(X_val[i, 0], **conf_plt)
    axes[ip, 1].pcolormesh(y_pred[i, 0], **conf_plt)
    im = axes[ip, 2].pcolormesh(y_val[i, 0], **conf_plt)
    if ip==0:
        axes[ip, 0].set_title("Low resolution")
        axes[ip, 1].set_title("Super-resolution")
        axes[ip, 2].set_title("Reference")

fig.colorbar(im, ax=axes, 
             orientation = 'horizontal',
             label="Transformed ash column load")